# Auto-MQT Token Routing Lab

Use this notebook for the proposal workflow: inspect examples, generate oracle labels, train prompt/image/multimodal routers, and compare against fixed-budget MQT-LLaVA baselines.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

ROOT

## Data Preparation

Prepare Hugging Face subsets into local JSONL manifests before running MQT-LLaVA. The manifest rows include `dataset`, `split`, `example_id`, `image`, `prompt`, `answer`, `answers`, and optional `task`.

```json
{"dataset": "textvqa", "split": "train", "example_id": "textvqa_train_12", "image": "data/images/textvqa/train/textvqa_train_12.jpg", "prompt": "what word is written on the sign?", "answer": "stop", "answers": ["stop"], "task": "ocr"}
```

In [ ]:
import json
import pandas as pd

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

# train_path = ROOT / "data" / "manifests" / "train.jsonl"
# eval_path = ROOT / "data" / "manifests" / "eval.jsonl"
# load_jsonl(train_path).head()

## Build Manifests

Start with tiny limits while checking dataset mirrors and schemas.

In [ ]:
# !python3 ../src/prepare_datasets.py --config ../configs/datasets.yaml --train-limit 5 --eval-limit 5 --prompt-style none
# !python3 ../src/verify_manifest.py --manifest ../data/manifests/train.jsonl
# !python3 ../src/verify_manifest.py --manifest ../data/manifests/eval.jsonl

## Fixed-Budget Baselines

These evaluate frozen MQT-LLaVA at a single visual-token budget.

In [ ]:
# !python3 ../src/evaluate_token_policy.py --data ../data/manifests/eval.jsonl --fixed-budget 8 --prompt-style short --out ../results/fixed_8.jsonl
# !python3 ../src/evaluate_token_policy.py --data ../data/manifests/eval.jsonl --fixed-budget 36 --prompt-style short --out ../results/fixed_36.jsonl
# !python3 ../src/evaluate_token_policy.py --data ../data/manifests/eval.jsonl --fixed-budget 256 --prompt-style short --out ../results/fixed_256.jsonl

## Oracle Label Generation

This is the expensive proposal step: run every training example across the MQT budget set and label it with the smallest sufficient budget.

In [ ]:
# !python3 ../src/oracle_labeling.py \
#   --data ../data/manifests/train.jsonl \
#   --out ../data/manifests/oracle_train_textvqa_small.jsonl \
#   --budgets 36 64 144 256 \
#   --score-key relaxed_match \
#   --prompt-style short \
#   --limit 10

## Train Routers

Train the prompt-only, image-only, and multimodal ablations from the proposal. The environment variable avoids a local macOS OpenMP import issue with PyTorch.

In [ ]:
# !KMP_DUPLICATE_LIB_OK=TRUE python3 ../src/train_router.py --labels ../data/manifests/oracle_train.jsonl --mode prompt --out ../checkpoints/router_prompt.pt
# !KMP_DUPLICATE_LIB_OK=TRUE python3 ../src/train_router.py --labels ../data/manifests/oracle_train.jsonl --mode image --out ../checkpoints/router_image.pt
# !KMP_DUPLICATE_LIB_OK=TRUE python3 ../src/train_router.py --labels ../data/manifests/oracle_train.jsonl --mode multimodal --out ../checkpoints/router_multimodal.pt

## Evaluate Learned Routing

In [ ]:
# !KMP_DUPLICATE_LIB_OK=TRUE python3 ../src/evaluate_router.py \
#   --data ../data/manifests/eval.jsonl \
#   --checkpoint ../checkpoints/router_multimodal.pt \
#   --out ../results/router_multimodal.jsonl

In [ ]:
def summarize_results(path):
    df = load_jsonl(path)
    return {
        "examples": len(df),
        "accuracy": df["exact_match"].mean(),
        "avg_visual_tokens": df["visual_tokens"].mean(),
        "avg_latency_s": df["latency_s"].mean(),
    }

# summarize_results(ROOT / "results" / "router_multimodal.jsonl")